In [1]:
import duckdb, os, shutil, json, time

SILVER = "silver"
GOLD   = "gold"
os.makedirs(GOLD, exist_ok=True)

con = duckdb.connect()

# views sobre a silver (fatos sao particionados por ano_mes -> hive_partitioning)
con.sql(f"""
CREATE OR REPLACE VIEW s_os  AS SELECT * FROM read_parquet('{SILVER}/ordens_servico/**/*.parquet', hive_partitioning=true);
CREATE OR REPLACE VIEW s_al  AS SELECT * FROM read_parquet('{SILVER}/alarmes_rede/**/*.parquet', hive_partitioning=true);
CREATE OR REPLACE VIEW s_med AS SELECT * FROM read_parquet('{SILVER}/medicoes_potencia/**/*.parquet', hive_partitioning=true);
CREATE OR REPLACE VIEW s_cto AS SELECT * FROM read_parquet('{SILVER}/ctos/*.parquet');
CREATE OR REPLACE VIEW s_tec AS SELECT * FROM read_parquet('{SILVER}/tecnicos/*.parquet');
CREATE OR REPLACE VIEW s_cli AS SELECT * FROM read_parquet('{SILVER}/clientes_operadoras/*.parquet');
CREATE OR REPLACE VIEW s_tip AS SELECT * FROM read_parquet('{SILVER}/tipos_servico/*.parquet');
CREATE OR REPLACE VIEW s_reg AS SELECT * FROM read_parquet('{SILVER}/dim_regiao/*.parquet');
""")

def grava(nome, sql, particao=None):
    """COPY sem PARTITION_BY escreve um ARQUIVO; com, escreve um diretorio."""
    t = time.time(); d = f"{GOLD}/{nome}"
    shutil.rmtree(d, ignore_errors=True)
    if particao:
        con.sql(f"COPY ({sql}) TO '{d}' (FORMAT PARQUET, PARTITION_BY ({particao}))")
    else:
        os.makedirs(d, exist_ok=True)
        con.sql(f"COPY ({sql}) TO '{d}/part-0.parquet' (FORMAT PARQUET)")
    n = con.sql(f"SELECT count(*) FROM read_parquet('{d}/**/*.parquet', hive_partitioning=true)").fetchone()[0]
    print(f"  {nome:24} {n:>7} linhas  ({time.time()-t:.1f}s)")

print("views criadas | OS na silver:", con.sql("SELECT count(*) FROM s_os").fetchone()[0])

views criadas | OS na silver: 50000


In [2]:
print("=== DIMENSOES ===")
grava("dim_tempo", """
SELECT d AS data, year(d) ano, month(d) mes, day(d) dia,
       strftime(d,'%Y-%m') ano_mes, 'T'||quarter(d) trimestre,
       week(d) semana_ano, dayofweek(d) dia_semana,
       strftime(d,'%A') nome_dia, strftime(d,'%B') nome_mes,
       dayofweek(d) IN (0,6) fim_de_semana
FROM (SELECT unnest(generate_series(DATE '2025-09-01', DATE '2026-08-31', INTERVAL 1 DAY)) d)""")

grava("dim_cliente", """
SELECT id_cliente, razao_social, nome_fantasia, cnpj,
       CAST(data_inicio_relacionamento AS DATE) data_inicio_relacionamento FROM s_cli""")

grava("dim_regiao", """
SELECT id_regiao, nome, cidade, uf,
       CAST(latitude_centro AS DOUBLE) latitude_centro,
       CAST(longitude_centro AS DOUBLE) longitude_centro FROM s_reg""")

grava("dim_tipo_servico", """
SELECT codigo_tipo_servico, descricao, grupo,
       CAST(duracao_padrao_min AS INT) duracao_padrao_min FROM s_tip""")

grava("dim_tecnico", """
SELECT id_tecnico, matricula, nome, nivel, equipe, id_regiao_base,
       CAST(data_admissao AS DATE) data_admissao,
       CAST(data_desligamento AS DATE) data_desligamento,
       data_desligamento IS NULL AS ativo FROM s_tec""")

grava("dim_cto", """
SELECT id_cto, id_olt, id_regiao, bairro_cto AS bairro, cidade_cto AS cidade,
       uf_cto AS uf, latitude, longitude FROM s_cto""")

=== DIMENSOES ===
  dim_tempo                    365 linhas  (0.0s)
  dim_cliente                    4 linhas  (0.0s)
  dim_regiao                     7 linhas  (0.0s)
  dim_tipo_servico               7 linhas  (0.0s)
  dim_tecnico                   41 linhas  (0.0s)
  dim_cto                     1200 linhas  (0.0s)


In [3]:
print("=== FATOS ===")
grava("fato_ordem_servico", """
SELECT id_os, numero_os, CAST(dt_abertura AS DATE) data_abertura,
       id_cliente, codigo_tipo_servico, id_tecnico, id_regiao, id_cto, id_assinante,
       canal_abertura, status, codigo_encerramento,
       dt_abertura, dt_despacho, dt_chegada, dt_encerramento,
       h_ate_despacho, h_deslocamento, h_execucao, h_atendimento,
       sla_horas, conta_para_sla, dentro_sla, horas_atraso,
       valor_os, multa_estimada, multa_no_teto,
       flag_datas_trocadas, flag_bairro_divergente, ano_mes
FROM s_os""", particao="ano_mes")

grava("fato_alarme_rede", """
SELECT id_alarme, CAST(dt_inicio AS DATE) data_inicio, id_cto, id_olt, id_regiao,
       porta_pon, id_assinante, tipo_alarme, severidade, severidade_nivel,
       dt_inicio, dt_fim, duracao_h, potencia_rx_dbm, qtd_onus_afetadas,
       flag_alarme_aberto, flag_nivel_rede, ano_mes
FROM s_al""", particao="ano_mes")

grava("fato_saude_cto", """
WITH al AS (SELECT id_cto, CAST(dt_inicio AS DATE) d, count(*) qtd_alarmes,
                   max(severidade_nivel) sev_max
            FROM s_al GROUP BY 1,2)
SELECT m.id_cto, m.data, m.potencia_rx_media_dbm, m.potencia_rx_min_dbm,
       m.onus_total, m.onus_online, m.pct_onus_online,
       m.pot_media_7d, m.pot_base_30d, m.delta_dbm, m.flag_dia_degradando,
       COALESCE(al.qtd_alarmes,0) qtd_alarmes, al.sev_max, m.ano_mes
FROM s_med m LEFT JOIN al ON al.id_cto=m.id_cto AND al.d=m.data""", particao="ano_mes")

=== FATOS ===
  fato_ordem_servico         50000 linhas  (0.7s)
  fato_alarme_rede           31574 linhas  (0.2s)
  fato_saude_cto            438000 linhas  (0.7s)


In [4]:
for v,t in [("f_os","fato_ordem_servico"),("f_al","fato_alarme_rede"),("f_cto","fato_saude_cto")]:
    con.sql(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM read_parquet('{GOLD}/{t}/**/*.parquet', hive_partitioning=true)")
for v,t in [("d_tempo","dim_tempo"),("d_cli","dim_cliente"),("d_tip","dim_tipo_servico"),
            ("d_reg","dim_regiao"),("d_cto","dim_cto"),("d_tec","dim_tecnico")]:
    con.sql(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM read_parquet('{GOLD}/{t}/*.parquet')")

print("=== INTEGRIDADE REFERENCIAL ===")
for fk,dim,ch in [("data_abertura","d_tempo","data"),("id_cliente","d_cli","id_cliente"),
                  ("codigo_tipo_servico","d_tip","codigo_tipo_servico"),("id_regiao","d_reg","id_regiao"),
                  ("id_cto","d_cto","id_cto"),("id_tecnico","d_tec","id_tecnico")]:
    r=con.sql(f"""SELECT count(*) FILTER (WHERE d.{ch} IS NULL AND f.{fk} IS NOT NULL) orfas,
                         count(*) FILTER (WHERE f.{fk} IS NULL) fk_nula
                  FROM f_os f LEFT JOIN {dim} d ON f.{fk}=d.{ch}""").fetchone()
    print(f"  {fk:22} orfas={r[0]:>4}  FK nula={r[1]:>4}")

=== INTEGRIDADE REFERENCIAL ===
  data_abertura          orfas=   0  FK nula=   0
  id_cliente             orfas=   0  FK nula=   0
  codigo_tipo_servico    orfas=   0  FK nula=   0
  id_regiao              orfas=   0  FK nula=   0
  id_cto                 orfas=   0  FK nula=   0
  id_tecnico             orfas=   0  FK nula=2968


In [5]:
gab = json.load(open('dados_telecom/_metadados.json', encoding='utf-8'))['gabarito']
ctos_gab = set(gab['ctos_problematicas']); tec_gab = set(gab['tecnicos_com_mais_reincidencia'])

# --- reincidencia por CTO (grao: ativo fisico, janela 30d)
con.sql("""CREATE OR REPLACE VIEW reinc_cto AS
WITH rep AS (SELECT id_os,id_cto,data_abertura FROM f_os
   WHERE codigo_tipo_servico IN ('REP_FTTH','REP_LENT','REP_ROMP') AND id_cto IS NOT NULL)
SELECT a.id_os, a.id_cto, count(b.id_os) reincidencias_30d
FROM rep a LEFT JOIN rep b ON b.id_cto=a.id_cto AND b.id_os<>a.id_os
 AND b.data_abertura < a.data_abertura AND b.data_abertura >= a.data_abertura - INTERVAL 30 DAY
GROUP BY 1,2""")

# --- retrabalho por TECNICO (grao: mesmo assinante, janela 7d)
con.sql("""CREATE OR REPLACE VIEW retrab_tec AS
WITH rep AS (SELECT id_os,id_assinante,id_tecnico,data_abertura,dt_encerramento FROM f_os
   WHERE codigo_tipo_servico IN ('REP_FTTH','REP_LENT','REP_ROMP')
     AND id_assinante IS NOT NULL AND id_tecnico IS NOT NULL AND status='CONCLUIDA')
SELECT a.id_tecnico, a.id_os,
  CASE WHEN EXISTS (SELECT 1 FROM rep b WHERE b.id_assinante=a.id_assinante AND b.id_os<>a.id_os
     AND b.data_abertura >  CAST(a.dt_encerramento AS DATE)
     AND b.data_abertura <= CAST(a.dt_encerramento AS DATE) + INTERVAL 7 DAY) THEN 1 ELSE 0 END rt
FROM rep a""")

grava("fato_reincidencia_cto", "SELECT * FROM reinc_cto")
grava("fato_retrabalho_tecnico", "SELECT * FROM retrab_tec")

print("=== CTOs problematicas ===")
top = [r[0] for r in con.sql("SELECT id_cto FROM f_cto GROUP BY 1 ORDER BY avg(pct_onus_online) ASC LIMIT 93").fetchall()]
print(f"  93 piores por pct_onus_online -> {sum(1 for c in top if c in ctos_gab)} das {len(ctos_gab)} do gabarito")

print("=== Tecnicos com mais retrabalho ===")
print(con.sql("""SELECT id_tecnico, count(*) reparos, sum(rt) retrabalho,
   round(avg(rt)*100,2) pct_retrabalho
 FROM retrab_tec GROUP BY 1 HAVING count(*)>=50 ORDER BY pct_retrabalho DESC LIMIT 6"""))

  fato_reincidencia_cto      23334 linhas  (0.4s)
  fato_retrabalho_tecnico    19627 linhas  (0.1s)
=== CTOs problematicas ===
  93 piores por pct_onus_online -> 90 das 91 do gabarito
=== Tecnicos com mais retrabalho ===
┌────────────┬─────────┬────────────┬────────────────┐
│ id_tecnico │ reparos │ retrabalho │ pct_retrabalho │
│  varchar   │  int64  │   int128   │     double     │
├────────────┼─────────┼────────────┼────────────────┤
│ TEC0018    │     661 │        100 │          15.13 │
│ TEC0011    │     656 │         99 │          15.09 │
│ TEC0007    │     699 │        104 │          14.88 │
│ TEC0033    │     267 │         36 │          13.48 │
│ TEC0022    │     277 │         36 │           13.0 │
│ TEC0026    │     536 │         69 │          12.87 │
└────────────┴─────────┴────────────┴────────────────┘



In [6]:
MAPA = """
WITH os_m AS (SELECT id_cto, ano_mes,
    count(*) os_total,
    count(*) FILTER (WHERE codigo_tipo_servico IN ('REP_FTTH','REP_LENT','REP_ROMP')) os_reparo,
    count(*) FILTER (WHERE conta_para_sla AND NOT dentro_sla) os_fora_sla,
    COALESCE(sum(multa_estimada),0) multa
  FROM f_os WHERE id_cto IS NOT NULL GROUP BY 1,2),
al_m AS (SELECT id_cto, ano_mes, count(*) alarmes,
    count(*) FILTER (WHERE tipo_alarme IN ('PON_DOWN','LOS','LOW_RX_POWER')) alarmes_preditivos,
    COALESCE(sum(qtd_onus_afetadas),0) onus_afetadas
  FROM f_al GROUP BY 1,2),
sa_m AS (SELECT id_cto, ano_mes, round(avg(pct_onus_online),2) pct_online,
    sum(CASE WHEN flag_dia_degradando THEN 1 ELSE 0 END) dias_degradando,
    round(min(pot_base_30d),2) pior_potencia
  FROM f_cto GROUP BY 1,2)
SELECT c.id_cto, s.ano_mes, c.id_regiao, c.bairro, c.cidade, c.uf, c.latitude, c.longitude,
  COALESCE(o.os_total,0) os_total, COALESCE(o.os_reparo,0) os_reparo,
  COALESCE(o.os_fora_sla,0) os_fora_sla, ROUND(COALESCE(o.multa,0),2) multa,
  COALESCE(a.alarmes,0) alarmes, COALESCE(a.alarmes_preditivos,0) alarmes_preditivos,
  COALESCE(a.onus_afetadas,0) onus_afetadas,
  s.pct_online, s.dias_degradando, s.pior_potencia,
  ROUND(100 - s.pct_online, 2) AS indice_risco
FROM sa_m s JOIN d_cto c USING (id_cto)
LEFT JOIN os_m o ON o.id_cto=s.id_cto AND o.ano_mes=s.ano_mes
LEFT JOIN al_m a ON a.id_cto=s.id_cto AND a.ano_mes=s.ano_mes"""

grava("fato_mapa_calor", MAPA, particao="ano_mes")

  fato_mapa_calor            14400 linhas  (0.3s)


In [7]:
RANK = f"""
WITH b AS (SELECT id_cto, any_value(id_regiao) id_regiao, any_value(bairro) bairro,
    any_value(cidade) cidade, any_value(uf) uf,
    any_value(latitude) latitude, any_value(longitude) longitude,
    round(avg(pct_online),2) pct_online_medio, sum(dias_degradando) dias_degradando,
    sum(alarmes) alarmes, sum(alarmes_preditivos) alarmes_preditivos,
    sum(os_reparo) os_reparo, sum(os_fora_sla) os_fora_sla, round(sum(multa),2) multa
  FROM read_parquet('{GOLD}/fato_mapa_calor/**/*.parquet', hive_partitioning=true) GROUP BY 1)
SELECT *, ROUND(100-pct_online_medio,2) indice_risco,
  RANK()  OVER (ORDER BY pct_online_medio ASC) rank_risco,
  NTILE(10) OVER (ORDER BY pct_online_medio ASC) decil_risco,
  pct_online_medio < 96.77 AS flag_cto_critica
FROM b"""

grava("dim_cto_risco", RANK)

  dim_cto_risco               1200 linhas  (0.4s)


In [8]:
LIFT = """
WITH rep AS (SELECT id_cto, data_abertura FROM f_os
             WHERE codigo_tipo_servico IN ('REP_FTTH','REP_LENT','REP_ROMP')),
base AS (SELECT 100.0*count(*) FILTER (WHERE EXISTS (SELECT 1 FROM rep r WHERE r.id_cto=d.id_cto
    AND r.data_abertura>=d.data AND r.data_abertura<=d.data+INTERVAL 3 DAY))/count(*) b FROM f_cto d)
SELECT a.tipo_alarme, a.severidade, count(*) alarmes,
  count(*) FILTER (WHERE EXISTS (SELECT 1 FROM rep r WHERE r.id_cto=a.id_cto
    AND r.data_abertura>=a.data_inicio AND r.data_abertura<=a.data_inicio+INTERVAL 3 DAY)) seguidos_de_reparo,
  ROUND(100.0*count(*) FILTER (WHERE EXISTS (SELECT 1 FROM rep r WHERE r.id_cto=a.id_cto
    AND r.data_abertura>=a.data_inicio AND r.data_abertura<=a.data_inicio+INTERVAL 3 DAY))/count(*),1) pct_conversao,
  ROUND((SELECT b FROM base),1) pct_base,
  ROUND((100.0*count(*) FILTER (WHERE EXISTS (SELECT 1 FROM rep r WHERE r.id_cto=a.id_cto
    AND r.data_abertura>=a.data_inicio AND r.data_abertura<=a.data_inicio+INTERVAL 3 DAY))/count(*))
    /(SELECT b FROM base),2) lift
FROM f_al a GROUP BY 1,2 ORDER BY lift DESC"""

grava("dim_alarme_preditivo", LIFT)
con.sql(f"SELECT * FROM read_parquet('{GOLD}/dim_alarme_preditivo/*.parquet')").show()

  dim_alarme_preditivo           5 linhas  (0.3s)
┌──────────────┬────────────┬─────────┬────────────────────┬───────────────┬──────────┬────────┐
│ tipo_alarme  │ severidade │ alarmes │ seguidos_de_reparo │ pct_conversao │ pct_base │  lift  │
│   varchar    │  varchar   │  int64  │       int64        │    double     │  double  │ double │
├──────────────┼────────────┼─────────┼────────────────────┼───────────────┼──────────┼────────┤
│ PON_DOWN     │ CRITICAL   │    1750 │               1750 │         100.0 │     18.1 │   5.52 │
│ LOS          │ MAJOR      │   10059 │               7753 │          77.1 │     18.1 │   4.26 │
│ LOW_RX_POWER │ WARNING    │    9787 │               5556 │          56.8 │     18.1 │   3.13 │
│ DYING_GASP   │ MINOR      │    6232 │               1907 │          30.6 │     18.1 │   1.69 │
│ HIGH_BER     │ WARNING    │    3746 │                972 │          25.9 │     18.1 │   1.43 │
└──────────────┴────────────┴─────────┴────────────────────┴───────────────┴─

In [9]:
# ============================================================
# PASSO FINAL - Publicar a gold como um banco unico
# ============================================================
DB = "telecom.duckdb"
if os.path.exists(DB):
    os.remove(DB)

pub = duckdb.connect(DB)
for tab in sorted(os.listdir(GOLD)):
    sub = [x for x in os.listdir(f"{GOLD}/{tab}") if os.path.isdir(f"{GOLD}/{tab}/{x}")]
    pat = f"{GOLD}/{tab}/**/*.parquet" if sub else f"{GOLD}/{tab}/*.parquet"
    pub.sql(f"CREATE OR REPLACE TABLE {tab} AS SELECT * FROM read_parquet('{pat}', hive_partitioning=true)")

print("tabelas publicadas:", [r[0] for r in pub.sql("SHOW TABLES").fetchall()])
print(f"tamanho: {os.path.getsize(DB)/1e6:.1f} MB")
pub.close()

tabelas publicadas: ['dim_alarme_preditivo', 'dim_cliente', 'dim_cto', 'dim_cto_risco', 'dim_regiao', 'dim_tecnico', 'dim_tempo', 'dim_tipo_servico', 'fato_alarme_rede', 'fato_mapa_calor', 'fato_ordem_servico', 'fato_reincidencia_cto', 'fato_retrabalho_tecnico', 'fato_saude_cto']
tamanho: 14.4 MB
